# Find and define unique peptides for all isoforms
In this notebook we will use the results from the tryptic in silico digestion and try to define a unique peptide for every isofofrm in our proteome.

# Setup
## Import and install Packages

In [ ]:
import sys
import os
import session_info

In [ ]:
import pandas as pd

In [ ]:
# Display session information
session_info.show()

## Set working directory

In [ ]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/02_In-Silico_Digestion"):
    print("WARNING: The working directory is not set to the '02_In-Silico_Digestion'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '02_In-Silico_Digestion' folder (\"{cwd}\").")

In [ ]:
# Data directory for fasta data
mapping_dir = "../01_UniProt/data/mapping/"
digestion_output_dir = "data/digestion_output"

## Read in Dataframes

In [ ]:
iso_canonical_mapping = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping.csv'))
iso_canonical_mapping_flat = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping_flat.csv'))

digestion_full_proteome_filtered =  pd.read_csv(os.path.join(digestion_output_dir, 'digestion_full_proteome_filtered.csv'))

# Find unique peptides for Isoforms

In [ ]:
canonical_ids = iso_canonical_mapping["Canonical_Isoform_ID"].dropna().tolist()
canonical_base_ids = [cid[:-2] if cid.endswith('-1') else cid for cid in canonical_ids]

In [ ]:
#filter canonical ID rows from iso_canonical_flat and from digestion_alternative_products_filtered
digestion_only_isoforms = digestion_full_proteome_filtered[~digestion_full_proteome_filtered["ID"].isin(canonical_base_ids)]
iso_canonical_filtered = iso_canonical_mapping_flat[~iso_canonical_mapping_flat["Isoform_ID"].isin(canonical_ids)]

In [ ]:
# Count how many unique Isoform IDs each peptide sequence belongs to
peptide_counts = digestion_only_isoforms.groupby('seq')['ID'].nunique()

# Identify sequences that appear in exactly ONE isoform
unique_sequences = peptide_counts[peptide_counts == 1].index

# Filter the original digestion dataframe to keep only these unique peptides
unique_peptides_df = digestion_only_isoforms[
    digestion_only_isoforms['seq'].isin(unique_sequences)
].copy()
unique_peptides_df

In [ ]:
# Isoforms that don't have a unique peptide
missing_isoforms = set(iso_canonical_filtered['Isoform_ID']) - set(unique_peptides_df['ID'])
print(f"Isoforms with no unique peptides: {len(missing_isoforms)}")

In [ ]:
# Redo with full proteome to make sure that there is no overlap with a canonical peptide
# Count how many unique Isoform IDs each peptide sequence belongs to
peptide_counts_fp = digestion_full_proteome_filtered.groupby('seq')['ID'].nunique()

# Identify sequences that appear in exactly ONE isoform
unique_sequences_fp = peptide_counts_fp[peptide_counts_fp == 1].index

# Filter the original digestion dataframe to keep only these unique peptides
unique_peptides_df_fp = digestion_full_proteome_filtered[
    digestion_full_proteome_filtered['seq'].isin(unique_sequences_fp)
].copy()
unique_peptides_df_fp

In [ ]:
# Number of isoforms for which we found a unique peptide
unique_peptides_df_fp["ID"].nunique()

In [ ]:
# Isoforms that don't have a unique peptide
all_ids = iso_canonical_flat['Isoform_ID'].tolist()

# Convert canonical ids to a set for much faster lookups (O(1) vs O(n))
canonical_set = set(canonical_ids)

# Update all_ids: strip -1 ONLY if the ID is in canonical_ids AND ends with -1
all_ids_updated = [
    aid[:-2] if (aid.endswith('-1') and aid in canonical_set) else aid 
    for aid in all_ids
]

missing_isoforms_fp = set(all_ids_updated) - set(unique_peptides_df_fp['ID'])
print(f"Isoforms with no unique peptides: {len(missing_isoforms_fp)}")

In [ ]:
# Sanity check: Are all proteins either in the unique_peptides df or in the missing_isoforms df?
print(unique_peptides_df_fp["ID"].nunique() + len(missing_isoforms_fp) == digestion_full_proteome_filtered["ID"].nunique())

In [ ]:
digestion_full_proteome_filtered["ID"].nunique()

In [ ]:
unique_peptides_df_fp["ID"].nunique() + len(missing_isoforms_fp)

In [ ]:
print(unique_peptides_df_fp["ID"].nunique() + len(missing_isoforms_fp) - digestion_full_proteome_filtered["ID"].nunique())